# Clasificador de noticias

En esta práctica crearemos un clasificador de noticias con técnicas de procesamiento del lenguaje natural, centrándonos en cómo representar el texto.

Utilizaremos el conjunto de datos [AG News](https://www.kaggle.com/amananandrai/ag-news-classification-dataset), con noticias de cuatro categorías.

## Conjunto de datos

Lo cargaremos con la biblioteca `datasets`, que permite acceder fácilmente a muchos conjuntos de datos. En este caso, cargaremos AG News.

## Preparación del conjunto de datos

Ejecuta la celda siguiente para instalar las bibliotecas necesarias. Utilizaremos `pytorch` para aprendizaje profundo, `scikit-learn` para aprendizaje automático y `transformers` para trabajar con modelos de lenguaje.

In [28]:
# Instalem les llibreries necessàries en les versions correctes

!pip install torch datasets scikit-learn transformers


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
from datasets import load_dataset

# Carreguem el dataset. Es descarregarà automàticament i es guardarà en local. 
# Aquest dataset conté notícies de diferents categories. En aquest cas
# utilitzarem les categories World, Sports, Business i Sci/Tech.

dataset = load_dataset('ag_news')

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [2]:
print(dataset['train'][0])

print(dataset['train'].features)

classes = dataset['train'].features["label"].names
classes

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


['World', 'Sports', 'Business', 'Sci/Tech']

La función `load` ha dividido automáticamente el conjunto en entrenamiento y prueba. Accederemos a ambos mediante los atributos `train` y `test` del objeto `dataset`. Cada partición contiene ejemplos y etiquetas; podemos consultarlos con los atributos `data` y `label`.

In [3]:
# Separem el dataset en conjunt d'entrenament i de test
ds_train = dataset['train']
ds_test = dataset['test']

# Vejam quants exemples hi ha en cada conjunt
print('Nombre d\'exemples de train:', len(ds_train))
print('Nombre d\'exemples de test:', len(ds_test))

Nombre d'exemples de train: 120000
Nombre d'exemples de test: 7600


Mostraremos los cinco primeros ejemplos del conjunto de entrenamiento. Cada ejemplo contiene una noticia y su etiqueta.

In [4]:
# Imprimim els primers 5 exemples del conjunt d'entrenament
for w in ds_train.take(5):
    print(f"{w['label']} ({classes[w['label']]}) -> {w['text']}")

2 (Business) -> Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2 (Business) -> Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
2 (Business) -> Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
2 (Business) -> Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.
2 (Business) -> Oil prices soar to

## Tokenización

Para que un modelo de lenguaje procese el texto, debemos convertirlo en números. Si queremos representarlo a nivel de palabra, necesitamos:

* Utilizar un **tokenizador** para dividir el texto en **tokens**.
* Construir un **vocabulario** a partir de esos tokens.

In [5]:
# Utilitzem el tokenitzador de BERT (un dels primers models de llenguatge basats en transformers) per tokenitzar les frases

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased")

print(tokenizer.tokenize("-- Hello, how are you doing today?"))

# Podem veure el vocabulari del tokenitzador
vocab = tokenizer.get_vocab()
print(len(vocab))

/home/carles/Documentos/notebooks/.venv/lib64/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


['-', '-', 'hello', ',', 'how', 'are', 'you', 'doing', 'today', '?']
30522


Con el tokenizador también podemos convertir la secuencia de tokens en números:

In [6]:
tokenitzada = tokenizer.tokenize("-- Hello, how are you doing today?")

def encode(text):
    tk = tokenizer.tokenize(text)
    return tokenizer.convert_tokens_to_ids(tk)

print(encode("-- Hello, how are you doing today?"))

[1011, 1011, 7592, 1010, 2129, 2024, 2017, 2725, 2651, 1029]


## Representación del texto

Para entrenar una red neuronal necesitamos representar el texto con vectores numéricos. En esta práctica utilizaremos *Bag of Words* (BoW), que representa cada palabra mediante un número. Es un método sencillo que no tiene en cuenta el orden ni la semántica de las palabras, pero funciona razonablemente bien en muchos casos.

### Representación *Bag of Words*

Sin contexto, a menudo es difícil deducir el significado de una palabra. Aun así, *Bag of Words* puede resultar útil. Por ejemplo, en una noticia, la palabra `covid` puede indicar que trata sobre la pandemia y `snow` puede sugerir que trata sobre el tiempo.

Una de las técnicas clásicas de vectorización más sencillas es *Bag of Words* (BoW). Cada palabra se representa mediante una posición del vector. Para convertir un texto, creamos primero un vector de ceros con una posición por cada palabra del vocabulario. Después, por cada palabra del texto, incrementamos el valor de su posición. Por ejemplo, para `this sentence is a test sentence`, el vector contará las apariciones de cada término.

Esta representación se parece a *one-hot*: en *one-hot* hay un vector por palabra, con un único 1; BoW suma esos vectores y cuenta las apariciones. La función `to_bow` que crearemos a continuación recibe un texto y devuelve su vector BoW.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
vectorizer.fit_transform(corpus)

vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

array([[1, 1, 0, 2, 0, 0, 0, 0, 0]])

Para calcular el vector BoW de una noticia de AG News, podemos utilizar la función siguiente:

In [8]:
import torch

len_vocab = len(vocab)

def to_bow(text, tamany_vocabulari=len_vocab):
    res = torch.zeros(tamany_vocabulari, dtype=torch.float32)

    for i in encode(text):
        if i<tamany_vocabulari:
            res[i] += 1
    return res

print(ds_train[0])
print(to_bow(ds_train[0]["text"]))

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
tensor([0., 0., 0.,  ..., 0., 0., 0.])


#### Entrenamiento del clasificador BoW

Nuestro primer modelo clasificará noticias mediante la representación BoW. Crearemos una red neuronal con una neurona de entrada por cada palabra del vocabulario y una neurona de salida por cada categoría.

#### Representación BoW

Primero convertiremos el texto con la función `to_bow`, que recibe un texto y devuelve su vector BoW.

En PyTorch, los `DataLoader` cargan los datos por lotes y los convierten en tensores. Usaremos el parámetro `collate_fn` para indicar cómo transformar los textos del lote en tensores BoW.

In [9]:
from torch.utils.data import DataLoader


# this collate function gets list of batch_size tuples, and needs to
# return a pair of label-feature tensors for the whole minibatch
def bowify(batch):
    """
    Aquesta funció rep una llista de noticies i retorna un tensor amb les etiquetes
    (vector de floats) i un altre amb les notícies codificades com a BoW (matriu de floats on cada fila
    és un vector de BoW).
    """

    # Els labels són 0, 1, 2 o 3.
    # Utilitzem LongTensor perquè són enters.

    etiquetes = torch.LongTensor([noticia["label"] for noticia in batch])

    # Les notícies són tensors de BoW
    noticies = torch.stack([to_bow(noticia["text"]) for noticia in batch])

    return (etiquetes, noticies)

train_loader = DataLoader(ds_train, batch_size=16, collate_fn=bowify)
test_loader = DataLoader(ds_test, batch_size=16, collate_fn=bowify)


#### Modelo de clasificación

Definamos una red neuronal sencilla con una capa lineal. El vector de entrada tiene tamaño `vocab_size` y la salida tiene cuatro valores, uno por clase. Como es un problema de clasificación, la activación final será `LogSoftmax()`.

In [10]:
net = torch.nn.Sequential(
    torch.nn.Linear(len(vocab), 4),
    torch.nn.LogSoftmax(dim=1)
)

# També podem definir la xarxa neuronal com a classe

class BoWClassifier(torch.nn.Module):
    def __init__(self, tamany_vocabulari, num_classes):
        # En el constructor definim les capes de la xarxa
        # crida al constructor de la classe pare
        super().__init__()

        # Capa lineal que passa de tamany_vocabulari a num_classes
        self.fc = torch.nn.Linear(tamany_vocabulari, num_classes)

        # Funció de softmax per obtenir probabilitats
        self.logsoftmax = torch.nn.LogSoftmax(dim=1)

        # Com a funció de pèrdua utilitzarem la NNLLLoss (Negative Log Likelihood Loss)
        # S'utilitza per classificació multiclasse i determina la pèrdua entre les prediccions
        # i les etiquetes

        self.loss = torch.nn.NLLLoss()

    def forward(self, ids):
        # x és el vector de BoW de la notícia
        # labels és l'etiqueta de la notícia

        # Passem x per la capa lineal
        ids = self.fc(ids)
        
        # Passem x per la funció de softmax
        ids = self.logsoftmax(ids)

        return ids
    
net = BoWClassifier(len(vocab), 4)

#### Entrenamiento del modelo

Ahora definiremos el bucle de entrenamiento habitual de PyTorch. Como el conjunto de datos es grande, para esta práctica entrenaremos durante una época o incluso menos. El parámetro `epoch_size` permite limitar el número de ejemplos procesados. También mostraremos la exactitud acumulada durante el entrenamiento; `report_freq` indica cada cuánto se informa.

Utilizaremos el optimizador `Adam` y la función de pérdida `CrossEntropyLoss`, adecuada para clasificar entre más de dos clases.

In [11]:
# Definim l'optimitzador, la funció de pèrdua i el dispositiu on es faran els càlculs
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn = torch.nn.CrossEntropyLoss()

In [12]:
def train_epoch(
    net,
    dataloader,
    device=None,
    optimizer=None,
    loss_fn=None,
    epoch_size=None,
    report_freq=200,
):
    # Movem la xarxa a la GPU
    net.to(device)

    # Posem la xarxa en mode training. Això activa el comportament de les capes Dropout, per exemple.
    net.train()

    # Inicialitzem les variables que ens serviran per calcular la precisió
    total_loss, acc, count, i = 0, 0, 0, 0

    # Iterem sobre el dataloader
    for labels, features in dataloader:
        # Movem les dades a la GPU
        labels, features = labels.to(device), features.to(device)

        # Posem els gradients a zero
        optimizer.zero_grad()

        # Calculem la sortida de la xarxa
        out = net(features)

        # Calculem la pèrdua. Aquesta funció ja aplica la softmax a la sortida.
        loss = loss_fn(out, labels)  # cross_entropy(out,labels)

        # Propaguem la pèrdua enrere. Això farà que es calculin els gradients.
        loss.backward()

        # Actualitzem els pesos de la xarxa. Això fa un pas d'optimització.
        optimizer.step()

        # Actualitzem les variables per calcular la precisió.
        total_loss += loss

        # Calculem la precisió. Per fer-ho, hem de convertir la sortida de la xarxa en etiquetes.
        # La classe amb la probabilitat més alta és la que predim com a etiqueta.
        _, predicted = torch.max(out, 1)
        acc += (predicted == labels).sum()

        # Actualitzem el comptador de mostres
        count += len(labels)

        # Mostrem la precisió cada report_freq mostres
        i += 1
        if i % report_freq == 0:
            print(f"{count}: acc={acc.item()/count}")

        # Si s'ha especificat epoch_size i ja hem processat aquest nombre de mostres, sortim del bucle.
        if epoch_size and count > epoch_size:
            break
    return total_loss.item() / count, acc.item() / count


# si volem entrenar la xarxa durant més èpoques, podem fer-ho amb un bucle
def train(
    net,
    train_loader,
    test_loader,
    device=None,
    optimizer=None,
    loss_fn=None,
    epochs=10,
    report_freq=200,
):
    if optimizer is None:
        optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

    if any(x is None for x in [device, loss_fn]):
        raise ValueError("device, optimizer and loss_fn must be specified")

    for epoch in range(epochs):
        print(f"Epoch {epoch}")
        train_loss, train_acc = train_epoch(
            net, train_loader, optimizer=optimizer, loss_fn=loss_fn, report_freq=report_freq
        )
        print(f"Train loss: {train_loss}, Train acc: {train_acc}")

In [13]:
train(net, train_loader, test_loader, device=device, loss_fn=loss_fn, epochs=1)

Epoch 0
3200: acc=0.7409375
6400: acc=0.80375
9600: acc=0.8259375
12800: acc=0.84125
16000: acc=0.8498125
19200: acc=0.8547916666666666
22400: acc=0.8603571428571428
25600: acc=0.8645703125
28800: acc=0.8661111111111112
32000: acc=0.868
35200: acc=0.8707102272727273
38400: acc=0.8723177083333333
41600: acc=0.8737019230769231
44800: acc=0.8752008928571429
48000: acc=0.8757916666666666
51200: acc=0.87615234375
54400: acc=0.8773345588235294
57600: acc=0.8781076388888889
60800: acc=0.8777302631578947
64000: acc=0.877859375
67200: acc=0.8788392857142857
70400: acc=0.879346590909091
73600: acc=0.8800815217391305
76800: acc=0.8815625
80000: acc=0.883325
83200: acc=0.8846995192307693
86400: acc=0.885011574074074
89600: acc=0.8852120535714286
92800: acc=0.8859482758620689
96000: acc=0.88646875
99200: acc=0.8866935483870968
102400: acc=0.887021484375
105600: acc=0.8877840909090909
108800: acc=0.8881801470588235
112000: acc=0.8884017857142857
115200: acc=0.8890104166666667
118400: acc=0.889518581

In [14]:
def validate(net, test_loader):
    net.eval()
    total_loss, acc, count = 0, 0, 0
    for labels, features in test_loader:
        out = net(features)
        _, predicted = torch.max(out, 1)
        acc += (predicted == labels).sum()
        count += len(labels)
    return acc.item() / count

In [15]:
validate(net, test_loader)

0.8925

El modelo alcanza una exactitud cercana a `0.9` en el conjunto de entrenamiento. Es un resultado razonable para este ejemplo, que hemos reducido para acortar el tiempo de ejecución. En una aplicación real utilizaríamos todas las noticias de entrenamiento y el modelo podría mejorar.

### Representación Word2Vec

Word2Vec es una representación habitual en procesamiento del lenguaje natural. Tiene en cuenta el contexto y permite realizar operaciones con vectores de palabras. Por ejemplo, al restar el vector de `man` al de `king` y sumar el de `woman`, obtenemos un vector parecido al de `queen`.

Utilizaremos `gensim` y el modelo `word2vec-google-news-300`, que contiene vectores para tres millones de palabras y frases.

> La primera vez que se ejecute esta celda, `load` descargará un archivo de 1,5 GB, lo que puede tardar unos minutos. Se guardará en `/home/USUARI/gensim-data/word2vec-google-news-300/word2vec-google-news-300.gz`, así que no será necesario volver a descargarlo.
> La función devuelve un objeto `KeyedVectors` con la representación Word2Vec.

In [16]:
import gensim.downloader as api

w2v = api.load('word2vec-google-news-300')

Ahora podemos consultar el vector Word2Vec de cada palabra. Por ejemplo, para obtener la representación de `king`, utilizaremos `get_vector` del objeto `KeyedVectors`.

In [17]:
w2v.get_vector('king')

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

También podemos buscar las palabras más parecidas a otra. Por ejemplo, utilizaremos `most_similar` para encontrar las más cercanas a `king`.

In [18]:
for w, p in w2v.most_similar('king'):
    print(f"{w} -> {p}")

kings -> 0.7138045430183411
queen -> 0.6510956883430481
monarch -> 0.6413194537162781
crown_prince -> 0.6204220056533813
prince -> 0.6159993410110474
sultan -> 0.5864824056625366
ruler -> 0.5797567367553711
princes -> 0.5646552443504333
Prince_Paras -> 0.5432944297790527
throne -> 0.5422105193138123


Lo más interesante de Word2Vec es que sus vectores tienen una estructura matemática que permite operar con palabras. Por ejemplo, al restar el vector de `man` al de `king` y sumar el de `woman`, obtenemos un vector próximo al de `queen`.

$$ KING - MAN + WOMAN = QUEEN $$

Para hacerlo utilizaremos `most_similar` de `KeyedVectors`, pasando los vectores de `king`, `woman` y `man`. La palabra más parecida al vector resultante debería ser `queen`.

In [19]:
w2v.most_similar(positive=['king', 'woman'], negative=['man'])[0]

('queen', 0.7118193507194519)

### Clasificador con Word2Vec

Ahora crearemos un clasificador de noticias con Word2Vec. Primero obtendremos el vector de cada palabra del texto y después sumaremos los vectores para obtener una representación de cada noticia.

La función `to_w2v` que definiremos a continuación recibe un texto y devuelve su vector Word2Vec.

In [20]:
def to_w2v(text):
    res = torch.zeros(300, dtype=torch.float32)
    for word in text:
        if word in w2v:
            res += torch.tensor(w2v.get_vector(word))
    return res

print(to_w2v(ds_train[0]["text"]))

tensor([-17.0809,  11.0404,  -0.9337,  12.4042,  -6.2286,   3.0224, -10.0442,
         -8.5156,  -5.9407,   1.1501,  -3.8471,  -8.0006, -18.2444,   4.3982,
        -14.2061,  11.0110,  11.2352,  14.8521,  -2.5686,   2.8961, -22.3914,
         -3.2182,   9.7872,   0.3238,  -8.6214,   4.2367, -21.9348,   5.7704,
         -0.6942,  -1.7075,  -2.4800,   2.1805,  -7.0602, -12.3824, -11.6949,
          8.2563, -18.9995,  11.3932,  -7.3198,   7.3370,  -6.1129,  -3.6244,
          5.8519,   8.3060,   3.9137,  -1.8091,  -3.2730, -15.8203,  -9.6418,
          8.9092, -16.8270,  24.5614,  -2.5387,  21.7112,   6.0571,  14.3324,
        -17.4978, -12.2693,   1.1129, -15.9192, -12.1886,  -9.5650, -19.0873,
         -7.7948,  -4.9111, -18.4653, -10.2332,  11.3437,  -6.0452,   5.4705,
          3.7500,  -9.5068,   4.4747,  -0.2912,  -3.9221,   0.3543,  13.0927,
          2.3088,   3.5300, -11.2126, -14.8031,  -2.9008,  -3.4219,  -0.3365,
         13.8353,   7.0914,  -5.2219,  22.0132,   4.2657,   5.84

Como hicimos con BoW, utilizaremos los `DataLoader` de PyTorch para convertir las representaciones Word2Vec en tensores. Usaremos el parámetro `collate_fn` para proporcionar la función que convierte los textos del lote en esos tensores.

In [21]:
def w2vify(batch):
    etiquetes = torch.LongTensor([noticia["label"] for noticia in batch])
    noticies = torch.stack([to_w2v(tokenizer.tokenize(noticia["text"])) for noticia in batch])
    return etiquetes, noticies

train_loader = DataLoader(ds_train, batch_size=16, collate_fn=w2vify)
test_loader = DataLoader(ds_test, batch_size=16, collate_fn=w2vify)

### Modelo de clasificación

Crearemos en PyTorch un modelo sencillo con una capa lineal. El vector de entrada tendrá tamaño `300`, que es la dimensión de Word2Vec, y la salida tendrá cuatro valores, uno por categoría. La función de activación final será `LogSoftmax()`.

In [22]:
net = torch.nn.Sequential(
    torch.nn.Linear(300, 4),
    torch.nn.LogSoftmax(dim=1)
)

Por último, entrenaremos el modelo con el mismo procedimiento que utilizamos para BoW.

In [23]:
train(net, train_loader, test_loader, device=device, loss_fn=loss_fn, epochs=5)

Epoch 0
3200: acc=0.7375
6400: acc=0.77109375
9600: acc=0.7835416666666667
12800: acc=0.79390625
16000: acc=0.8014375
19200: acc=0.80484375
22400: acc=0.8094642857142857
25600: acc=0.81375
28800: acc=0.8157986111111111
32000: acc=0.819625
35200: acc=0.821875
38400: acc=0.8231770833333333
41600: acc=0.8246153846153846
44800: acc=0.8258705357142857
48000: acc=0.8262708333333333
51200: acc=0.8261328125
54400: acc=0.8265257352941177
57600: acc=0.8272743055555556
60800: acc=0.8262828947368421
64000: acc=0.826078125
67200: acc=0.8270833333333333
70400: acc=0.827684659090909
73600: acc=0.8289130434782609
76800: acc=0.8301692708333334
80000: acc=0.832925
83200: acc=0.8349879807692308
86400: acc=0.835625
89600: acc=0.8365513392857142
92800: acc=0.8374030172413793
96000: acc=0.8376979166666667
99200: acc=0.8374798387096775
102400: acc=0.837607421875
105600: acc=0.8384469696969697
108800: acc=0.8388602941176471
112000: acc=0.8390892857142858
115200: acc=0.8398524305555556
118400: acc=0.8404138513

In [24]:
validate(net, test_loader)

0.8497368421052631

El resultado no es muy bueno porque el modelo Word2Vec utilizado no contiene algunas palabras del conjunto de datos. Por ejemplo, `covid` no aparece en el vocabulario del modelo.

Para resolverlo, podríamos utilizar un modelo entrenado con las palabras de nuestro conjunto o uno con un vocabulario mayor. Lo veremos en el siguiente ejemplo.

## Redes neuronales recurrentes

Por último, utilizaremos redes neuronales recurrentes (RNN) para clasificar noticias. Las RNN procesan secuencias; en este caso, procesarán las palabras de cada noticia.

### Carga de datos

Crearemos un `DataLoader` que organice los datos como secuencias de palabras. Con `collate_fn` indicaremos cómo convertir los textos en secuencias de índices. En este caso no convertiremos las palabras en vectores Word2Vec.

In [30]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    texts = [torch.tensor(encode(item["text"])) for item in batch]
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    return labels, texts_padded


# Creem els DataLoader
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(ds_test, batch_size=16, shuffle=False, collate_fn=collate_fn)

### Modelo de clasificación

Crearemos en PyTorch un modelo sencillo con una capa RNN y una capa de salida, con una neurona por cada categoría. La activación final será `LogSoftmax()`.

No podemos utilizar `Sequential`, porque hay que gestionar el estado interno de la RNN. Por eso definiremos la clase `RNNClassifier`, heredada de `nn.Module`, e implementaremos el método `forward`.

También utilizaremos `nn.Embedding` para convertir los índices de las palabras en vectores que pueda procesar la RNN.

In [31]:
import torch.nn as nn

class RNNClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        # Inicialitzem els atributs del model
        # Hidden dim és la dimensió de l'estat ocult de la RNN
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.rnn = torch.nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x,h = self.rnn(x)
        return self.fc(x.mean(dim=1))
    
# Paràmetres del model
vocab_size = len(tokenizer.get_vocab())  # Mida del vocabulari
embedding_dim = 64  # Dimensió dels embeddings
hidden_dim = 32  # Dimensió de l'estat ocult de la LSTM
num_class = len(classes)  # Nombre de classes (World, Sports, Business, Sci/Tech)

# Creem el model
model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, num_class)

In [32]:
train(model, train_loader, test_loader, device=device, loss_fn=loss_fn, epochs=5)

Epoch 0
3200: acc=0.508125
6400: acc=0.65078125
9600: acc=0.7163541666666666
12800: acc=0.750078125
16000: acc=0.7735625
19200: acc=0.7906770833333333
22400: acc=0.8041071428571429
25600: acc=0.814453125
28800: acc=0.8213541666666667
32000: acc=0.82815625
35200: acc=0.8335511363636363
38400: acc=0.83859375
41600: acc=0.8425961538461538
44800: acc=0.8465625
48000: acc=0.8498541666666667
51200: acc=0.85291015625
54400: acc=0.8555330882352942
57600: acc=0.8583159722222222
60800: acc=0.8605756578947369
64000: acc=0.86240625
67200: acc=0.8638988095238095
70400: acc=0.8652130681818182
73600: acc=0.8667119565217392
76800: acc=0.8684375
80000: acc=0.86955
83200: acc=0.8711177884615384
86400: acc=0.8720138888888889
89600: acc=0.8731473214285714
92800: acc=0.8742025862068965
96000: acc=0.8751875
99200: acc=0.87625
102400: acc=0.877109375
105600: acc=0.8778787878787879
108800: acc=0.8786305147058824
112000: acc=0.8794375
115200: acc=0.8802690972222222
118400: acc=0.8808108108108108
Train loss: 0.

El mejor resultado se obtiene con la representación de una red recurrente LSTM: una exactitud cercana a `0.94`, sin tener que calcular previamente los vectores de las palabras.

El modelo solo tiene una capa recurrente y una lineal, pero podríamos probar más capas, *dropout* y otros ajustes.

Antes de la aparición de los Transformers, las RNN eran las arquitecturas más utilizadas para procesar lenguaje natural y aún se usan en distintos casos. En la siguiente práctica veremos cómo utilizar Transformers para obtener mejores resultados.